# DIAMOND — خدمة استبدال/تحريك الشخصية بالفيديو (Wan2.2-Animate)
تشغّل هذه الدفتر نموذج **Wan2.2-Animate** المفتوح على كرت Colab، وتعطيك واجهة برابط عام مؤقت:
- **وضع الاستبدال:** تضع شخصية من صورة مكان الشخص في فيديو المشهد (بحركته).
- **وضع التحريك:** تحرّك صورة شخصية بحركة شخص في فيديو.

**المتطلبات (مهم):**
- كرت كبير: **A100 (40 أو 80 جيجابايت)** أو أعلى. كرت T4 المجاني لا يكفي. يتطلب Colab Pro/Pay-as-you-go أو استئجار GPU مثل RunPod.
- مساحة نحو 60 جيجابايت وإنترنت سريع لتنزيل النموذج أول مرة.
- التوليد بطيء: دقائق لكل بضع ثوانٍ من الفيديو.
- هذا الدفتر لم أجرّبه على كرت حقيقي، فقد يحتاج تعديلًا إن تغيّر مستودع Wan2.2.

**الاستخدام المسؤول:** لا ترفع صورة وجه أو فيديو شخص دون إذنه، واستخدم شخصياتك أو شخصًا أذن لك.

In [ ]:
import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "لا يوجد — غيّر نوع التشغيل إلى GPU")
if torch.cuda.is_available(): print("الذاكرة (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9,1))

In [ ]:
%cd /content
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
!pip -q install -r requirements.txt || true
!pip -q install flash_attn --no-build-isolation || echo "flash_attn لم يُثبّت، سيعمل بدونه"
!pip -q install -r requirements_animate.txt
!pip -q install "huggingface_hub[cli]" gradio

In [ ]:
%cd /content/Wan2.2
!huggingface-cli download Wan-AI/Wan2.2-Animate-14B --local-dir ./Wan2.2-Animate-14B

In [ ]:
import os, subprocess, glob, time, shutil, gradio as gr
os.chdir("/content/Wan2.2")
WORK = "/content/work"

def sh(cmd, log):
    log.append("$ " + " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    log.append(out[-3000:])
    return p.returncode

def run(video, image, mode, res, save_mem, ok):
    log = []
    if not ok: raise gr.Error("أكّد أن لديك إذنًا باستخدام الصورة والفيديو.")
    if not video or not image: raise gr.Error("ارفع الفيديو والصورة.")
    job = os.path.join(WORK, str(int(time.time()))); proc = os.path.join(job, "proc"); os.makedirs(proc, exist_ok=True)
    w, h = (1280, 720) if res == "720p" else (832, 480)
    pre = ["python", "./wan/modules/animate/preprocess/preprocess_data.py", "--ckpt_path", "./Wan2.2-Animate-14B/process_checkpoint",
           "--video_path", video, "--refer_path", image, "--save_path", proc, "--resolution_area", str(w), str(h)]
    if mode.startswith("استبدال"): pre += ["--iterations", "3", "--k", "7", "--w_len", "1", "--h_len", "1", "--replace_flag"]
    else: pre += ["--retarget_flag"]
    if sh(pre, log) != 0: raise gr.Error("فشلت المعالجة المسبقة:\n" + "\n".join(log)[-2500:])
    out = os.path.join(job, "out.mp4")
    gen = ["python", "generate.py", "--task", "animate-14B", "--ckpt_dir", "./Wan2.2-Animate-14B/", "--src_root_path", proc + "/", "--refert_num", "1", "--save_file", out]
    if mode.startswith("استبدال"): gen += ["--replace_flag", "--use_relighting_lora"]
    if save_mem: gen += ["--offload_model", "True", "--convert_model_dtype", "--t5_cpu"]
    rc = sh(gen, log)
    if not os.path.exists(out):
        cand = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getmtime)
        cand = [c for c in cand if os.path.getmtime(c) > time.time() - 7200 and "process_results" not in c and not c.startswith(proc)]
        if cand: out = cand[-1]
    if rc != 0 and not os.path.exists(out): raise gr.Error("فشل التوليد:\n" + "\n".join(log)[-2500:])
    return out, "\n".join(log)[-3000:]

with gr.Blocks(title="DIAMOND Wan-Animate") as demo:
    gr.Markdown("### استبدال/تحريك الشخصية — Wan2.2-Animate")
    with gr.Row():
        v = gr.Video(label="فيديو الأداء / المشهد"); i = gr.Image(type="filepath", label="صورة الشخصية")
    mode = gr.Radio(["استبدال شخصية في الفيديو", "تحريك صورة بحركة الفيديو"], value="استبدال شخصية في الفيديو", label="الوضع")
    res = gr.Radio(["480p", "720p"], value="480p", label="الدقة (ابدأ بـ 480p)")
    mem = gr.Checkbox(True, label="توفير الذاكرة (أبطأ لكنه أأمن)")
    ok = gr.Checkbox(False, label="أؤكد أن لدي إذنًا باستخدام هذه الصورة وهذا الفيديو")
    go = gr.Button("🚀 ابدأ", variant="primary")
    o = gr.Video(label="النتيجة"); lg = gr.Textbox(label="السجل", lines=8)
    go.click(run, [v, i, mode, res, mem, ok], [o, lg])
demo.queue().launch(share=True, debug=True)